# 03 — Training artifact review
Inspect the existing Phase 4 training artifacts and training-only preprocessing. This notebook reuses the trained models; it does **not** fit, tune, serialize or replace them. The frozen Phase 3 data and all PDFs remain unchanged.

In [1]:
from pathlib import Path
import os, sys, json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ml/data/phase3_dataset_manifest.json').is_file())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
approved = Path(os.environ['LOCALAPPDATA']) / 'SmartParking/ml-runtime/Scripts/python.exe'
assert Path(sys.executable).resolve() == approved.resolve()
import numpy as np
import pandas as pd
from ml.features import FEATURE_COLUMNS
from ml.phase4_models import load_predictor, sha256
from ml.train_phase4 import read_partition, verify_manifest
manifest = json.loads(Path('ml/data/phase3_dataset_manifest.json').read_text(encoding='utf-8'))
results = json.loads(Path('ml/artifacts/phase4/model_results.json').read_text(encoding='utf-8'))
lock_path = Path('ml/artifacts/phase4/validation_selection.json')
lock = json.loads(lock_path.read_text(encoding='utf-8'))
verify_manifest(manifest)
assert sha256(lock_path) == results['selection_lock_sha256']
bundle = load_predictor(results['artifact']['path'], results['artifact']['sha256'])
print('Interpreter:', sys.executable)
print('Trusted artifact SHA256:', results['artifact']['sha256'])

Interpreter: C:\Users\vighn\AppData\Local\SmartParking\ml-runtime\Scripts\python.exe
Trusted artifact SHA256: cfc2adc80cc949e3c097530bca500079b857eee8b931be1e6d7de522f50335e1


## Frozen training design
Persistence predicts the observed current total. Linear Regression minimizes training squared error after training-only StandardScaler transformation. Random Forest averages bootstrapped regression trees; a four-candidate depth/leaf grid was compared on validation data. All three models use the same 37 predictors.
Models were not refitted on validation. Selection minimizes clipped validation MAE, with RMSE then simplicity as tie-breakers. The selection JSON was locked before the original test evaluation. Reading that saved decision here does not perform a new selection.

In [2]:
x_train, y_train, train_info = read_partition(manifest, 'train')
assert train_info == results['partitions']['train']
linear = bundle.models['linear_regression']
scaler = linear.named_steps['scale']
regressor = linear.named_steps['regressor']
assert int(scaler.n_samples_seen_) == len(x_train) == 49360
np.testing.assert_allclose(scaler.mean_, x_train.mean(axis=0), rtol=0, atol=1e-10)
np.testing.assert_allclose(scaler.var_, x_train.var(axis=0), rtol=0, atol=1e-10)
assert lock['tuning']['refit_on_validation'] is False
assert results['models_refitted_on_validation'] is False
assert bundle.metadata['trained_on'] == 'train_only'
assert len(scaler.mean_) == len(FEATURE_COLUMNS) == 37
print('PASS: scaler mean, variance and sample count agree with training rows only')
print('Linear rank:', regressor.rank_, 'of', len(FEATURE_COLUMNS))
display(pd.DataFrame({'feature': FEATURE_COLUMNS, 'training_mean': scaler.mean_, 'training_scale': scaler.scale_, 'standardized_coefficient': regressor.coef_}).head(15))

PASS: scaler mean, variance and sample count agree with training rows only
Linear rank: 34 of 37


,feature,training_mean,training_scale,standardized_coefficient
0,P1,0.223683,0.416712,0.256777
1,P2,0.190904,0.393013,0.262809
2,P3,0.211629,0.408463,0.257443
3,P4,0.184299,0.387728,0.261707
4,P5,0.166147,0.372212,0.228077
5,P6,0.175041,0.380002,0.249115
6,P7,0.217220,0.412354,0.230771
7,P8,0.210413,0.407602,0.240169
8,P9,0.212075,0.408777,0.255153
9,P10,0.186122,0.389205,0.245432


In [3]:
forest = bundle.models['random_forest']
chosen = lock['tuning']['chosen_forest_parameters']
assert all(forest.get_params()[key] == value for key, value in chosen.items())
assert len(forest.estimators_) == chosen['n_estimators'] == 100
assert bundle.metadata['seed'] == manifest['seed'] == 42
assert bundle.metadata['dataset_sha256'] == manifest['processed_file']['sha256']
candidates = lock['tuning']['forest_candidates']
assert len(candidates) == 4
display(pd.DataFrame([{'candidate': c['candidate'], 'max_depth': c['parameters']['max_depth'], 'min_samples_leaf': c['parameters']['min_samples_leaf'], 'validation_mae': c['validation']['clipped']['mae'], 'validation_rmse': c['validation']['clipped']['rmse']} for c in candidates]))
display(pd.Series(chosen, name='Saved forest configuration').to_frame())
print('PASS: saved forest matches validation-selected configuration')
print('Selected model:', bundle.selected_model, '| test used for selection:', results['test_used_for_selection'])

,candidate,max_depth,min_samples_leaf,validation_mae,validation_rmse
0,0,8,10,0.445964,0.695580
1,1,8,30,0.436999,0.680876
2,2,12,10,0.484498,0.729381
3,3,12,30,0.458959,0.700878


,Saved forest configuration
n_estimators,100
max_depth,8
min_samples_leaf,30
max_features,0.8
max_samples,0.8
bootstrap,True
random_state,42
n_jobs,2


PASS: saved forest matches validation-selected configuration
Selected model: persistence | test used for selection: False


## Evidence and limitations
A checksum protects against accidental changes, not malicious replacement of a joblib file; load only this trusted repository artifact. Linear rank deficiency reflects correlated slot totals/features; it does not justify rewriting frozen predictors. Forest impurity importance is not causal attribution.
The frozen report records actual local fitting and a later verification-only roundoff correction (absolute tolerance 1e−12) using the unchanged bundle. This notebook demonstrates artifact consistency, not fresh training or real-world accuracy.